# LightGCN — Ablation Study & Results

This notebook reproduces and extends the experiments from **Table 3** and **Section 4.3**
of the LightGCN paper (He et al., SIGIR 2020).

We investigate three questions:
1. **How does the number of graph convolution layers (K) affect performance?**
2. **Does using all-layer combination beat using only the last layer?**
3. **How do our results compare to the numbers reported in the paper?**

Each experiment trains a separate LightGCN model from scratch on the Gowalla dataset.
Evaluation: **Recall@20** and **NDCG@20** using the all-ranking protocol (Section 4.1).

> **Runtime note:** Running all experiments end-to-end takes ~3 h on Kaggle GPU.
> Each K-value is trained for 100 epochs. Use Run All, then walk away.
>
> **Why 100 epochs?** Empirical runs on Gowalla show the model reaches ~95% of its
> converged performance by epoch 100. The biggest gains happen before epoch 60;
> improvement becomes marginal after epoch 120–130. K-curve separation is already
> clearly visible by epoch 50.

## Setup

Path detection works in both environments:
- **Kaggle**: uses `/kaggle/working` as project root, data from `/kaggle/input/datasets/jackkozx/gowalla-dataset`
- **VS Code**: uses the `__vsc_ipynb_file__` global to locate the project root

We also detect the best available device (CUDA → MPS → CPU).

In [ ]:
import os
from pathlib import Path

# On Kaggle: clone repo to get src/ modules (skipped if src/ already exists)
if Path("/kaggle").exists() and not Path("src").exists():
    os.system("git clone --depth=1 https://github.com/JacKoz7/LightGCN-Recommender /tmp/lgcn")
    os.system("cp -r /tmp/lgcn/src .")

In [ ]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# --- path detection (VS Code + Kaggle) ---
if Path("/kaggle").exists():
    PROJECT_ROOT = Path("/kaggle/working")
    DATA_PATH    = Path("/kaggle/input/datasets/jackkozx/gowalla-dataset")
else:
    _nb_file = globals().get("__vsc_ipynb_file__")
    if _nb_file:
        PROJECT_ROOT = Path(_nb_file).resolve().parent.parent
    else:
        PROJECT_ROOT = Path(os.getcwd())
        if PROJECT_ROOT.name == "notebooks":
            PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_PATH = PROJECT_ROOT / "data" / "gowalla"

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dataset  import GowallaDataset
from model    import LightGCN
from evaluate import recall_and_ndcg_at_k

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Project root : {PROJECT_ROOT}")
print(f"Data path    : {DATA_PATH}")
print(f"Device       : {device}")

## Load dataset

We load the Gowalla dataset once and reuse it across all experiments.
Building the normalized adjacency matrix `Ã = D^(-1/2) A D^(-1/2)` is the most
expensive setup step — there is no point repeating it for every configuration.

In [ ]:
print("Loading dataset ...")
dataset  = GowallaDataset(DATA_PATH)
norm_adj = dataset.norm_adj.to(device)

print(f"Users             : {dataset.n_users:>8,}")
print(f"Items (places)    : {dataset.n_items:>8,}")
print(f"Train interactions: {len(dataset.train_pairs):>8,}")
print(f"Test users        : {len(dataset.test_user_items):>8,}")

## Training helper

`run_experiment` runs a full training loop for a given model configuration and
returns a **history dictionary** — loss, Recall@20, and NDCG@20 recorded at every
evaluation checkpoint. This makes it easy to plot convergence curves and compare
configurations without duplicating boilerplate.

The BPR loss implementation is identical to `src/train.py`:
- Sample `(user, positive item, negative item)` triples
- Push the positive score above the negative via `log σ(score_pos − score_neg)`
- Add L2 regularization **only on the 0th-layer embeddings** of the sampled nodes (Eq. 15)

In [3]:
def _sample_batch(train_user_items, n_items, batch_size):
    all_users = list(train_user_items.keys())
    users, pos_items, neg_items = [], [], []
    while len(users) < batch_size:
        user = random.choice(all_users)
        pos  = random.choice(train_user_items[user])
        neg  = random.randint(0, n_items - 1)
        while neg in train_user_items[user]:
            neg = random.randint(0, n_items - 1)
        users.append(user)
        pos_items.append(pos)
        neg_items.append(neg)
    return (torch.LongTensor(users),
            torch.LongTensor(pos_items),
            torch.LongTensor(neg_items))


def run_experiment(model_cls=LightGCN, n_layers=3, emb_dim=64,
                   lr=0.001, reg_lambda=1e-4, batch_size=1024,
                   n_epochs=200, eval_every=10, label=None):
    """Train a LightGCN model and return (history, best_recall, best_ndcg)."""
    label = label or f"{model_cls.__name__}, K={n_layers}"
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")

    model = model_cls(
        n_users=dataset.n_users, n_items=dataset.n_items,
        emb_dim=emb_dim, n_layers=n_layers, norm_adj=norm_adj,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    n_batches = max(1, len(dataset.train_pairs) // batch_size)
    history   = {"epoch": [], "loss": [], "recall": [], "ndcg": []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss = 0.0

        for _ in range(n_batches):
            u, pi, ni = _sample_batch(dataset.train_user_items, dataset.n_items, batch_size)
            u, pi, ni = u.to(device), pi.to(device), ni.to(device)

            optimizer.zero_grad()
            user_emb, item_emb = model()
            uu, ii, jj = user_emb[u], item_emb[pi], item_emb[ni]
            loss = -torch.log(torch.sigmoid((uu * ii).sum(-1) - (uu * jj).sum(-1))).mean()

            e0  = model.embedding.weight
            reg = (e0[u].norm(2).pow(2)
                   + e0[dataset.n_users + pi].norm(2).pow(2)
                   + e0[dataset.n_users + ni].norm(2).pow(2)) / len(u)
            (loss + reg_lambda * reg).backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % eval_every == 0:
            recall, ndcg = recall_and_ndcg_at_k(
                model, dataset.train_user_items, dataset.test_user_items, k=20
            )
            history["epoch"].append(epoch)
            history["loss"].append(total_loss / n_batches)
            history["recall"].append(recall)
            history["ndcg"].append(ndcg)
            print(f"  Epoch {epoch:>4} | Loss {total_loss/n_batches:.4f} "
                  f"| Recall@20 {recall:.4f} | NDCG@20 {ndcg:.4f}")

    best_recall = max(history["recall"]) if history["recall"] else 0.0
    best_ndcg   = history["ndcg"][history["recall"].index(best_recall)] if history["recall"] else 0.0
    print(f"  → Best Recall@20: {best_recall:.4f}   Best NDCG@20: {best_ndcg:.4f}")
    return history, best_recall, best_ndcg

---

## Experiment 1 — Effect of the number of layers K

The key hyperparameter in LightGCN is **K** — how many times we propagate embeddings
through the graph before computing the final representation.

- **K=1**: each node only sees its direct neighbors
- **K=2**: each node also sees neighbors-of-neighbors (similar users who visited the same places)
- **K=3**: one hop further — the places those similar users visited (candidate recommendations)
- **K=4**: one more hop — can cause over-smoothing (all embeddings start to look alike)

The paper reports K=3 as optimal on Gowalla (Table 3). We replicate this.

All other hyperparameters are fixed: `emb_dim=64, lr=0.001, λ=1e-4, batch_size=1024`.

In [ ]:
results_k = {}
for k in [1, 2, 3, 4]:
    history, best_recall, best_ndcg = run_experiment(
        n_layers=k, n_epochs=150, eval_every=5, label=f"K={k} layers"
    )
    results_k[k] = {"history": history, "recall": best_recall, "ndcg": best_ndcg}

### Visualisation: convergence curves and best performance per K

**Left plot** — how Recall@20 evolves over training for each K.  
**Right plot** — best Recall@20 achieved per K (reproduces Figure 3 from the paper).

At full convergence (~1000 epochs) the paper reports: K=3 > K=2 > K=4 > K=1.  
At **150 epochs** (our ablation), K=2 leads because it converges faster than K=3.  
The drop at K=4 is over-smoothing — confirmed in both the paper and our run.


In [ ]:
colors = ["#4c72b0", "#dd8452", "#55a868", "#c44e52"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: convergence curves
ax = axes[0]
for (k, res), color in zip(results_k.items(), colors):
    ax.plot(res["history"]["epoch"], res["history"]["recall"],
            label=f"K={k}", color=color, linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Recall@20")
ax.set_title("Recall@20 convergence for different K")
ax.legend()
ax.grid(True, alpha=0.3)

# Right: bar chart of best Recall@20 per K
ax = axes[1]
ks      = list(results_k.keys())
recalls = [results_k[k]["recall"] for k in ks]
bars    = ax.bar([str(k) for k in ks], recalls, color=colors, width=0.5)
for bar, val in zip(bars, recalls):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f"{val:.4f}", ha="center", va="bottom", fontsize=9)
ax.set_xlabel("Number of layers K")
ax.set_ylabel("Best Recall@20")
ax.set_title("Best Recall@20 by number of layers")
ax.set_ylim(0, max(recalls) * 1.12)
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
(PROJECT_ROOT / "notebooks").mkdir(exist_ok=True)
plt.savefig(PROJECT_ROOT / "notebooks" / "k_layer_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

best_k = max(results_k, key=lambda k: results_k[k]["recall"])
print(f"\nBest K: {best_k}  (Recall@20 = {results_k[best_k]['recall']:.4f})")

---

## Experiment 2 — Layer combination vs. last-layer-only

LightGCN uses a **uniform average** of all K+1 layer outputs as the final embedding (Eq. 4):

```
e_final = (E^0 + E^1 + E^2 + E^3) / 4
```

An alternative is to use **only the last layer** E^K (like most GCN variants do).

Why does the paper prefer the average?
- **E^0 preserves identity** — without it, the model loses the node's original "self" representation
- **E^1, E^2, E^3 add neighbourhood context** at increasing distances
- **Averaging prevents over-smoothing** — using only E^3 risks all embeddings converging
  to similar values, losing the ability to distinguish individual users/items

We test this claim by comparing two variants with K=3:
- `LightGCN` — our existing model with all-layer combination (Eq. 4)
- `LastLayerLightGCN` — uses only E^3 as the final embedding

In [ ]:
class LastLayerLightGCN(LightGCN):
    """LightGCN variant that uses only the final propagation layer — no layer combination."""
    def forward(self):
        E = self.embedding.weight
        for _ in range(self.n_layers):
            E = torch.sparse.mm(self.norm_adj, E)
        return E[:self.n_users], E[self.n_users:]

In [ ]:
history_all,  recall_all,  ndcg_all  = run_experiment(
    model_cls=LightGCN, n_layers=3, n_epochs=100, eval_every=5,
    label="LightGCN K=3, all-layer combination"
)
history_last, recall_last, ndcg_last = run_experiment(
    model_cls=LastLayerLightGCN, n_layers=3, n_epochs=100, eval_every=5,
    label="LightGCN K=3, last-layer only"
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_all["epoch"],  history_all["recall"],
        label="All-layer combination (Eq. 4)", linewidth=2, color="#4c72b0")
ax.plot(history_last["epoch"], history_last["recall"],
        label="Last layer only", linewidth=2, linestyle="--", color="#c44e52")
ax.set_xlabel("Epoch")
ax.set_ylabel("Recall@20")
ax.set_title("All-layer combination vs. last-layer-only  (K=3)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
(PROJECT_ROOT / "notebooks").mkdir(exist_ok=True)
plt.savefig(PROJECT_ROOT / "notebooks" / "layer_combination.png", dpi=150, bbox_inches="tight")
plt.show()

improvement = (recall_all - recall_last) / max(recall_last, 1e-9) * 100
print(f"All-layer combination : Recall@20 = {recall_all:.4f}   NDCG@20 = {ndcg_all:.4f}")
print(f"Last-layer only       : Recall@20 = {recall_last:.4f}   NDCG@20 = {ndcg_last:.4f}")
print(f"Improvement from layer combination: {improvement:+.1f}%")

---

## Experiment 3 — Comparison with paper results (Table 3)

The paper (He et al., SIGIR 2020) reports the following numbers for Gowalla, K=3, emb_dim=64:

| | Recall@20 | NDCG@20 |
|---|---|---|
| Paper (Table 3) | 0.1823 | 0.1554 |

We compare two of our results against the paper:
- **K=3 from Experiment 1 (150 epochs)** — the ablation-study result
- **Full 400-epoch run (K=3, `src/train.py`)** — our best single-model result

The 150-epoch gap (~6.7%) closes to **~2.7%** at 400 epochs (Recall@20 = 0.1773).
The remaining gap is explained by the paper training for up to 1000 epochs.


In [ ]:
paper_recall = 0.1823
paper_ndcg   = 0.1554

our_recall = results_k[3]["recall"]
our_ndcg   = results_k[3]["ndcg"]

gap_recall = (paper_recall - our_recall) / paper_recall * 100
gap_ndcg   = (paper_ndcg   - our_ndcg)   / paper_ndcg   * 100

print(f"{'Configuration':40} {'Recall@20':>10} {'NDCG@20':>10}")
print("-" * 62)
print(f"{'Paper (He et al., SIGIR 2020)  K=3':40} {paper_recall:>10.4f} {paper_ndcg:>10.4f}")
print(f"{'Our implementation              K=3':40} {our_recall:>10.4f} {our_ndcg:>10.4f}")
print("-" * 62)
print(f"{'Gap vs. paper':40} {gap_recall:>+9.1f}% {gap_ndcg:>+9.1f}%")

### Full comparison table across all K values

In [ ]:
# K=3 comes from Table 3 in the paper (exact).
# K=1,2,4 are read off Figure 3 in the paper (approximate — the figure shows
# relative trends, not exact values, so treat these as indicative).
paper_numbers = {
    1: {"recall": 0.1549, "ndcg": 0.1312},
    2: {"recall": 0.1736, "ndcg": 0.1477},
    3: {"recall": 0.1823, "ndcg": 0.1554},
    4: {"recall": 0.1821, "ndcg": 0.1547},
}

print(f"{'K':>3} | {'Paper Recall@20':>16} | {'Ours Recall@20':>15} | {'Paper NDCG@20':>14} | {'Ours NDCG@20':>13}")
print("-" * 70)
for k in [1, 2, 3, 4]:
    pr  = paper_numbers[k]["recall"]
    pn  = paper_numbers[k]["ndcg"]
    or_ = results_k[k]["recall"]
    on  = results_k[k]["ndcg"]
    print(f"{k:>3} | {pr:>16.4f} | {or_:>15.4f} | {pn:>14.4f} | {on:>13.4f}")

#
#
 
S
u
m
m
a
r
y


*
*
E
x
p
e
r
i
m
e
n
t
 
1
 
—
 
E
f
f
e
c
t
 
o
f
 
K
 
(
1
5
0
-
e
p
o
c
h
 
r
u
n
)
:
*
*


|
 
K
 
|
 
B
e
s
t
 
R
e
c
a
l
l
@
2
0
 
|
 
B
e
s
t
 
N
D
C
G
@
2
0
 
|
 
P
e
a
k
 
e
p
o
c
h
 
|

|
-
-
-
|
-
-
-
|
-
-
-
|
-
-
-
|

|
 
1
 
|
 
0
.
1
6
6
3
 
|
 
0
.
1
4
2
8
 
|
 
1
3
5
 
|

|
 
*
*
2
*
*
 
|
 
*
*
0
.
1
7
1
2
*
*
 
|
 
*
*
0
.
1
4
7
0
*
*
 
|
 
1
5
0
 
(
s
t
i
l
l
 
r
i
s
i
n
g
)
 
|

|
 
3
 
|
 
0
.
1
7
0
1
 
|
 
0
.
1
4
5
6
 
|
 
1
4
5
 
|

|
 
4
 
|
 
0
.
1
6
6
3
 
|
 
0
.
1
4
1
9
 
|
 
1
4
5
 
|


K
=
2
 
w
i
n
s
 
a
t
 
1
5
0
 
e
p
o
c
h
s
.
 
N
o
t
a
b
l
y
,
 
K
=
2
 
h
a
d
 
*
*
n
o
t
 
y
e
t
 
c
o
n
v
e
r
g
e
d
*
*
 
a
t
 
e
p
o
c
h
 
1
5
0
 
—
 
i
t
s

R
e
c
a
l
l
@
2
0
 
w
a
s
 
s
t
i
l
l
 
i
n
c
r
e
a
s
i
n
g
 
m
o
n
o
t
o
n
i
c
a
l
l
y
,
 
s
u
g
g
e
s
t
i
n
g
 
i
t
 
w
o
u
l
d
 
c
o
n
t
i
n
u
e
 
t
o
 
i
m
p
r
o
v
e
.

K
=
3
 
p
e
a
k
e
d
 
a
t
 
e
p
o
c
h
 
1
4
5
 
a
n
d
 
d
r
o
p
p
e
d
 
s
l
i
g
h
t
l
y
 
a
t
 
1
5
0
,
 
m
e
a
n
i
n
g
 
i
t
 
i
s
 
c
l
o
s
e
 
t
o
 
c
o
n
v
e
r
g
e
n
c
e
.


T
h
e
 
g
a
p
 
b
e
t
w
e
e
n
 
K
=
2
 
a
n
d
 
K
=
3
 
i
s
 
o
n
l
y
 
0
.
0
0
1
1
 
—
 
w
i
t
h
i
n
 
t
h
e
 
n
o
i
s
e
 
o
f
 
a
 
s
i
n
g
l
e
 
r
u
n
.

T
h
e
 
p
a
p
e
r
 
r
e
p
o
r
t
s
 
K
=
3
 
a
s
 
o
p
t
i
m
a
l
 
a
t
 
1
0
0
0
 
e
p
o
c
h
s
;
 
w
i
t
h
 
m
o
r
e
 
t
r
a
i
n
i
n
g
 
K
=
3
 
w
o
u
l
d
 
l
i
k
e
l
y

c
a
t
c
h
 
u
p
,
 
c
o
n
s
i
s
t
e
n
t
 
w
i
t
h
 
t
h
e
 
p
a
p
e
r
'
s
 
f
i
n
d
i
n
g
.


T
h
e
 
k
e
y
 
p
a
t
t
e
r
n
 
t
h
e
 
p
a
p
e
r
 
p
r
e
d
i
c
t
s
 
*
*
d
o
e
s
 
h
o
l
d
*
*
:

-
 
M
o
r
e
 
l
a
y
e
r
s
 
h
e
l
p
 
u
p
 
t
o
 
a
 
p
o
i
n
t
 
(
K
=
2
 
>
 
K
=
1
)

-
 
K
=
4
 
c
l
e
a
r
l
y
 
o
v
e
r
-
s
m
o
o
t
h
s
 
—
 
i
t
 
e
q
u
a
l
s
 
K
=
1
 
i
n
 
f
i
n
a
l
 
R
e
c
a
l
l
@
2
0
 
d
e
s
p
i
t
e
 
s
l
o
w
e
r
 
c
o
n
v
e
r
g
e
n
c
e

-
 
T
h
e
 
o
p
t
i
m
a
l
 
K
 
s
i
t
s
 
i
n
 
t
h
e
 
2
–
3
 
r
a
n
g
e


*
*
E
x
p
e
r
i
m
e
n
t
 
2
 
—
 
L
a
y
e
r
 
c
o
m
b
i
n
a
t
i
o
n
:
*
*

A
v
e
r
a
g
i
n
g
 
a
l
l
 
l
a
y
e
r
 
o
u
t
p
u
t
s
 
(
E
q
.
 
4
)
 
c
o
n
s
i
s
t
e
n
t
l
y
 
o
u
t
p
e
r
f
o
r
m
s
 
u
s
i
n
g
 
o
n
l
y
 
t
h
e
 
f
i
n
a
l
 
l
a
y
e
r
.

T
h
e
 
0
t
h
 
l
a
y
e
r
 
(
o
r
i
g
i
n
a
l
 
e
m
b
e
d
d
i
n
g
s
)
 
a
c
t
s
 
a
s
 
a
 
r
e
g
u
l
a
r
i
s
e
r
 
t
h
a
t
 
p
r
e
s
e
r
v
e
s
 
n
o
d
e
 
i
d
e
n
t
i
t
y

a
n
d
 
p
r
e
v
e
n
t
s
 
t
h
e
 
r
e
p
r
e
s
e
n
t
a
t
i
o
n
s
 
f
r
o
m
 
c
o
l
l
a
p
s
i
n
g
.


*
*
E
x
p
e
r
i
m
e
n
t
 
3
 
—
 
C
o
m
p
a
r
i
s
o
n
 
w
i
t
h
 
p
a
p
e
r
:
*
*

O
u
r
 
K
=
3
 
a
t
 
1
5
0
 
e
p
o
c
h
s
:
 
R
e
c
a
l
l
@
2
0
 
=
 
0
.
1
7
0
1
 
v
s
.
 
p
a
p
e
r
 
0
.
1
8
2
3
 
(
g
a
p
:
 
−
6
.
7
%
)
.
 
 

O
u
r
 
K
=
2
 
a
t
 
1
5
0
 
e
p
o
c
h
s
:
 
R
e
c
a
l
l
@
2
0
 
=
 
0
.
1
7
1
2
 
v
s
.
 
p
a
p
e
r
 
K
=
2
 
o
f
 
0
.
1
7
3
6
 
(
g
a
p
:
 
−
1
.
4
%
)
.
 
 

O
u
r
 
K
=
3
 
*
*
f
u
l
l
 
4
0
0
-
e
p
o
c
h
 
r
u
n
*
*
:
 
R
e
c
a
l
l
@
2
0
 
=
 
*
*
0
.
1
7
7
3
*
*
 
v
s
.
 
p
a
p
e
r
 
0
.
1
8
2
3
 
(
g
a
p
:
 
−
*
*
2
.
7
%
*
*
)
.
 
 

N
o
t
a
b
l
y
,
 
o
u
r
 
K
=
1
 
(
0
.
1
6
6
3
)
 
a
n
d
 
K
=
2
 
(
0
.
1
7
1
2
)
 
a
t
 
1
5
0
 
e
p
o
c
h
s
 
*
*
e
x
c
e
e
d
*
*
 
t
h
e
 
p
a
p
e
r
'
s
 
K
=
1
 
(
0
.
1
5
4
9
)

a
n
d
 
c
o
m
e
 
w
i
t
h
i
n
 
1
.
4
%
 
o
f
 
p
a
p
e
r
'
s
 
K
=
2
 
(
0
.
1
7
3
6
)
 
—
 
c
o
n
f
i
r
m
i
n
g
 
f
a
i
t
h
f
u
l
 
i
m
p
l
e
m
e
n
t
a
t
i
o
n
.


T
h
e
 
c
e
n
t
r
a
l
 
c
l
a
i
m
 
o
f
 
L
i
g
h
t
G
C
N
 
—
 
*
*
r
e
m
o
v
i
n
g
 
f
e
a
t
u
r
e
 
t
r
a
n
s
f
o
r
m
a
t
i
o
n
 
a
n
d
 
n
o
n
l
i
n
e
a
r
 
a
c
t
i
v
a
t
i
o
n

m
a
k
e
s
 
G
C
N
-
b
a
s
e
d
 
r
e
c
o
m
m
e
n
d
a
t
i
o
n
 
s
i
m
p
l
e
r
,
 
f
a
s
t
e
r
,
 
a
n
d
 
m
o
r
e
 
a
c
c
u
r
a
t
e
*
*
 
—
 
h
o
l
d
s
 
i
n
 
o
u
r
 
r
e
p
r
o
d
u
c
t
i
o
n
.